In [ ]:
from ml4cps import examples, tools, vis
discrete_data, cont_data = examples.conveyor_system_sfowl(split=True)

In [ ]:
discrete_data, discrete_data_valid, discrete_data_test = discrete_data
cont_data, cont_data_valid, cont_data_test = cont_data

## Continuous data


In [ ]:
mode_data = tools.encode_columns_to_string([x for x in discrete_data[0:9:4]])
mode_data[0]

In [ ]:
fig = vis.plot_timeseries(cont_data[0:9:4], mode_data=mode_data, mode_height=0.2,
                    use_columns=['LH1_power', 'LH2_power', 'RH2_power', 'RH1_power', 'RV_power',
                                 'LV_power'], x_title="Time [s]").update_layout(height=800, width=800, font_color="black")
fig
# fig.write_image('conveyor_system_sfowl_ts.svg', height=800, width=800)

In [ ]:
len(cont_data)

In [ ]:
from ml4cps import vis
vis.plot_timeseries(cont_data[0:3]).update_layout(height=1200).show()

In [ ]:
from ml4cps import debta
import torch

columns = cont_data[0].columns
train_data = debta.WindowedSequenceDataset(cont_data)
valid_data = debta.WindowedSequenceDataset(cont_data_valid)

In [ ]:
mean, std = train_data.normalize()
valid_data.normalize(mean=mean, std=std)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = debta.DEBTA(num_y=24, num_h=10, first_hidden_size=50, num_sigm_layers=2, sigma=0.3, device=device)

model.pretrain_layers(train_data=train_data, valid_data=valid_data, lr=[0.001, 0.01], verbose=False, max_epoch=50, batch_size=128)



In [ ]:
model.learn_latent_automaton(train_dataset=train_data, valid_dataset=valid_data)

In [ ]:
model.num_modes

In [ ]:
vis.plot_cps_component(model, output='notebook', dash_port=12345)

In [ ]:
model.remove_rare_transitions(min_num=3)
vis.plot_cps_component(model, output='notebook', dash_port=12346)

In [ ]:
model.num_modes

In [ ]:
vis.plot_cps_component(model, output='notebook', node_labels=True, event_label=False, center_node_labels=True, show_transition_data=False, show_transition_timing=True, freq_as_edge_thickness=True, max_zoom=5, show_transition_freq=True, dash_port=12345)